In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import requests

In [3]:
project_path = ".."

raw_path = os.path.join(project_path, "data", "raw")
cleaned_path = os.path.join(project_path, "data", "cleaned")
plots_path = os.path.join(project_path, "plots")

print("Project:", os.path.abspath(project_path))
print("Raw data:", os.path.abspath(raw_path))
print("Cleaned data:", os.path.abspath(cleaned_path))
print("Plots:", os.path.abspath(plots_path))

Project: /home/darshani/lightkurve-env/quannt
Raw data: /home/darshani/lightkurve-env/quannt/data/raw
Cleaned data: /home/darshani/lightkurve-env/quannt/data/cleaned
Plots: /home/darshani/lightkurve-env/quannt/plots


In [4]:
os.makedirs(raw_path, exist_ok=True)
os.makedirs(cleaned_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

print("All project folders are ready!")

All project folders are ready!


In [7]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "30"
}

print("API URL:", url)
print("Currency:", params["vs_currency"])
print("Days:", params["days"])

API URL: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart
Currency: usd
Days: 30


In [8]:
response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

Status code: 200


In [9]:
if response.status_code == 200:
    data = response.json()

    print("Data downloaded successfully!")
    print("Available data categories:", list(data.keys()))
else:
    print("Request failed:", response.text)

Data downloaded successfully!
Available data categories: ['prices', 'market_caps', 'total_volumes']


In [10]:
import json

raw_file_path = os.path.join(raw_path, "bitcoin_raw.json")

with open(raw_file_path, "w") as file:
    json.dump(data, file, indent=4)

print("Raw data saved successfully!")
print("File location:", raw_file_path)

Raw data saved successfully!
File location: ../data/raw/bitcoin_raw.json


In [11]:
print("File exists:", os.path.exists(raw_file_path))

print(
    "File size:",
    round(os.path.getsize(raw_file_path) / 1024, 2),
    "KB"
)

File exists: True
File size: 165.57 KB


In [12]:
with open(raw_file_path, "r") as file:
    raw_data = json.load(file)

print("Data categories:", list(raw_data.keys()))
print("\nFirst 3 price records:")
print(raw_data["prices"][:3])

Data categories: ['prices', 'market_caps', 'total_volumes']

First 3 price records:
[[1790103600000, 86476.73261291615], [1790107200000, 86220.71105203898], [1790110800000, 86250.64221068537]]


In [13]:
prices = raw_data["prices"]

print("Total price records:", len(prices))
print("\nFirst 5 records:")
print(prices[:5])

Total price records: 721

First 5 records:
[[1790103600000, 86476.73261291615], [1790107200000, 86220.71105203898], [1790110800000, 86250.64221068537], [1790114400000, 86187.63078333685], [1790118000000, 86120.19453430604]]


In [14]:
price_df = pd.DataFrame(
    prices,
    columns=["timestamp", "price_usd"]
)

price_df.head()

,timestamp,price_usd
0,1790103600000,86476.732613
1,1790107200000,86220.711052
2,1790110800000,86250.642211
3,1790114400000,86187.630783
4,1790118000000,86120.194534


In [15]:
price_df["timestamp"] = pd.to_datetime(
    price_df["timestamp"],
    unit="ms",
    utc=True
)

price_df.head()

,timestamp,price_usd
0,2026-09-22 19:00:00+00:00,86476.732613
1,2026-09-22 20:00:00+00:00,86220.711052
2,2026-09-22 21:00:00+00:00,86250.642211
3,2026-09-22 22:00:00+00:00,86187.630783
4,2026-09-22 23:00:00+00:00,86120.194534


In [16]:
print("Dataset shape:", price_df.shape)

print("\nColumn names:")
print(price_df.columns)

print("\nData types:")
print(price_df.dtypes)

print("\nDataset information:")
price_df.info()

Dataset shape: (721, 2)

Column names:
Index(['timestamp', 'price_usd'], dtype='object')

Data types:
timestamp    datetime64[ns, UTC]
price_usd                float64
dtype: object

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 721 entries, 0 to 720
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   timestamp  721 non-null    datetime64[ns, UTC]
 1   price_usd  721 non-null    float64            
dtypes: datetime64[ns, UTC](1), float64(1)
memory usage: 11.4 KB


In [17]:
print("Missing values in each column:")
print(price_df.isnull().sum())

Missing values in each column:
timestamp    0
price_usd    0
dtype: int64


In [18]:
print("Duplicate rows:", price_df.duplicated().sum())

Duplicate rows: 0


In [19]:
cleaned_df = price_df.copy()

# Remove rows with missing timestamps or prices
cleaned_df = cleaned_df.dropna(
    subset=["timestamp", "price_usd"]
)

# Remove duplicate rows
cleaned_df = cleaned_df.drop_duplicates()

# Remove rows with invalid prices
cleaned_df = cleaned_df[
    cleaned_df["price_usd"] > 0
]

# Sort observations by timestamp
cleaned_df = cleaned_df.sort_values(
    "timestamp"
)

# Reset the row index
cleaned_df = cleaned_df.reset_index(
    drop=True
)

print("Cleaning completed!")
print("Original rows:", len(price_df))
print("Cleaned rows:", len(cleaned_df))

Cleaning completed!
Original rows: 721
Cleaned rows: 721


In [20]:
cleaned_file_path = os.path.join(
    cleaned_path,
    "bitcoin_cleaned.csv"
)

cleaned_df.to_csv(
    cleaned_file_path,
    index=False
)

print("Cleaned data saved successfully!")
print("Location:", cleaned_file_path)

Cleaned data saved successfully!
Location: ../data/cleaned/bitcoin_cleaned.csv


In [22]:
import pyarrow
cleaned_file_path = os.path.join(
    cleaned_path,
    "bitcoin_cleaned.parquet"
)

cleaned_df.to_parquet(
    cleaned_file_path,
    index=False,
    engine="pyarrow"
)

print("Cleaned data saved successfully!")
print("File location:", cleaned_file_path)

Cleaned data saved successfully!
File location: ../data/cleaned/bitcoin_cleaned.parquet
